<p> <center> <a href="../start_here_ja.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="02_introduction_mcp_ja.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="01_inference_endpoint_ja.ipynb">1</a>
        <a href="02_introduction_mcp_ja.ipynb">2</a>
        <a >3</a>
        <a href="04_langraph_agent_ja.ipynb">4</a>
        <a href="05_nemo_agent_toolkit_ja.ipynb">5</a>
        <a href="06_challenge_ja.ipynb">6</a>
        <a href="bonus_challenge/07_bonus_challenge_ja.ipynb">7</a>
    </span>
    <span style="float: left; width: 33%; text-align: right;"><a href="04_langraph_agent_ja.ipynb">Next Notebook</a></span>
</div>

## 学習目標

このノートブックを終えると、参加者は以下ができるようになります：

* high-level (FastMCP) と low-level MCP SDK の違いを理解する
* low-level SDK を使って明示的なツール定義で MCP server を構築する
* `types.Tool` と JSON Schema を使ってツールスキーマを手動で定義する
* input validation とエラーハンドリングを含むカスタムツールハンドラーを実装する
* Stdio と HTTP の両方の transport を使って low-level MCP server をデプロイする
* MCP client を使って low-level server に接続する

## Low-Level MCP Server の実装
前のノートブックでは、**FastMCP**（high-level SDK）を使って素早く MCP server を構築しました。次は、より冗長なコードと引き換えにサーバーの動作を完全に制御できる **low-level SDK** を探っていきましょう。

各アプローチをいつ使うかを以下に示します：

| 機能 | High-Level (FastMCP) | Low-Level SDK |
|---------|---------------------|---------------|
| **ツール定義** | `@mcp.tool()` デコレーター | 手動で `types.Tool` schema を定義 |
| **Input Validation** | type hints から自動 | 手動 validation が必要 |
| **ボイラープレート** | 最小限 | より冗長 |
| **制御** | 抽象化済み | フルコントロール |
| **ユースケース** | クイックプロトタイピング | 本番環境、カスタム動作 |

### Stdio Transport を使った Low-Level MCP Server

low-level SDK を使って MCP server を構築してみましょう。デコレーターを使うだけの FastMCP と異なり、ここでは以下が必要です：

1. **`Server` インスタンスの作成** – コアとなる server オブジェクト
2. **`@server.list_tools()` の定義** – JSON Schema を使って tool schema を手動で指定する
3. **`@server.call_tool()` の定義** – 手動 validation でツールの実行を処理する
4. **transport のセットアップ** – 通信用の stdio ストリームを初期化する

In [ ]:
%%writefile mcp_server_low_level.py
"""
Low-Level MCP Server Implementation (Stdio Transport)

This server demonstrates how to build MCP servers with full control
over tool definitions, input validation, and error handling.
"""

import asyncio
import logging
from typing import Any

from mcp.server import InitializationOptions
from mcp.server.lowlevel import Server, NotificationOptions
from mcp.server.stdio import stdio_server
import mcp.types as types

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('simple-math')


async def main():
    # ===========================================
    # Step 1: Create the Server Instance
    # ===========================================
    server = Server("simple-math")

    # ===========================================
    # Step 2: Define Available Tools
    # ===========================================
    @server.list_tools()
    async def handle_list_tools() -> list[types.Tool]:
        """
        Returns the name, description, and input schema of all available tools.
        
        Each tool requires:
        - name: Unique identifier for the tool
        - description: Human-readable description
        - inputSchema: JSON Schema defining expected arguments
        """
        return [
            types.Tool(
                name="add",
                description="Add two numbers together",
                inputSchema={
                    "type": "object",
                    "properties": {
                        "a": {"type": "number", "description": "First number"},
                        "b": {"type": "number", "description": "Second number"}
                    },
                    "required": ["a", "b"],
                },
            ),
            types.Tool(
                name="subtract",
                description="Subtract second number from first",
                inputSchema={
                    "type": "object",
                    "properties": {
                        "a": {"type": "number", "description": "First number"},
                        "b": {"type": "number", "description": "Second number"}
                    },
                    "required": ["a", "b"]
                },
            ),
        ]

    # ===========================================
    # Step 3: Implement Tool Handlers
    # ===========================================
    @server.call_tool()
    async def handle_call_tool(
        name: str, arguments: dict[str, Any] | None
    ) -> list[types.TextContent | types.ImageContent | types.EmbeddedResource]:
        """
        Handle tool execution requests.
        
        Args:
            name: The name of the tool to execute
            arguments: Dictionary of arguments passed to the tool
            
        Returns:
            List of content objects (TextContent, ImageContent, etc.)
        """
        try:
            # Validate arguments exist
            if not arguments:
                raise ValueError("No arguments provided")
            
            # Handle 'add' tool
            if name == "add":
                if "a" not in arguments or "b" not in arguments:
                    raise ValueError("Missing required arguments: a, b")
                result = arguments['a'] + arguments['b']
                return [types.TextContent(type="text", text=str(result))]

            # Handle 'subtract' tool
            elif name == "subtract":
                if "a" not in arguments or "b" not in arguments:
                    raise ValueError("Missing required arguments: a, b")
                result = arguments['a'] - arguments['b']
                return [types.TextContent(type="text", text=str(result))]
            
            else:
                raise ValueError(f"Unknown tool: {name}")
                
        except Exception as e:
            logger.error(f"Error executing tool '{name}': {e}")
            return [types.TextContent(type="text", text=f"Error: {str(e)}")]

    # ===========================================
    # Step 4: Initialize and Run the Server
    # ===========================================
    async with stdio_server() as (read_stream, write_stream):
        logger.info("Server running with stdio transport")
        await server.run(
            read_stream,
            write_stream,
            InitializationOptions(
                server_name="simple-math",
                server_version="0.1.0",
                capabilities=server.get_capabilities(
                    notification_options=NotificationOptions(),
                    experimental_capabilities={},
                ),
            ),
        )


# Entry point
if __name__ == "__main__":
    asyncio.run(main())

low-level server に接続してツールを呼び出すために MCP client を使いましょう。

In [ ]:
import subprocess
import time

# Start the HTTP server in the background
server_process = subprocess.Popen(
    ["python", "mcp_server_low_level.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for server to start
time.sleep(3)

# Store and display the process ID
SERVER_PID = server_process.pid
print(f"Process ID: {SERVER_PID}")

クライアントを実行してツールを接続・呼び出してみましょう。

In [ ]:
!python mcp_client.py mcp_server_low_level.py

完了したら必ずバックグラウンドプロセスをクリーンアップしてください。

In [ ]:
!kill -9 {SERVER_PID}
print(f"✓ Server (PID: {SERVER_PID}) killed")

### HTTP Transport を使った Low-Level MCP Server

本番環境へのデプロイやリモートアクセスには、Stdio の代わりに **HTTP transport** を使用できます。server は環境変数 `MCP_PORT` から port を読み取り、未設定の場合は `8000` にフォールバックします。そのため、デフォルトでは `http://localhost:8000/mcp` でアクセスできます。

In [ ]:
%%writefile mcp_server_low_level_http.py
"""
Low-Level MCP Server Implementation (HTTP Transport)

Uses Starlette + Uvicorn for production-ready HTTP deployment.
Endpoint: http://localhost:${MCP_PORT:-8000}/mcp
"""

import asyncio
import logging
import os
import contextlib
from collections.abc import AsyncIterator
from typing import Any

from mcp.server import InitializationOptions
from mcp.server.lowlevel import Server, NotificationOptions
from mcp.server.streamable_http_manager import StreamableHTTPSessionManager
from starlette.applications import Starlette
from starlette.routing import Mount
from starlette.types import Receive, Scope, Send
import uvicorn
import mcp.types as types

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('simple-math')
MCP_PORT = int(os.environ.get("MCP_PORT") or "8000")


def main():
    # ===========================================
    # Step 1: Create the Server Instance
    # ===========================================
    server = Server("simple-math")

    # ===========================================
    # Step 2: Define Available Tools
    # ===========================================
    @server.list_tools()
    async def handle_list_tools() -> list[types.Tool]:
        """List available tools with their schemas."""
        return [
            types.Tool(
                name="add",
                description="Add two numbers together",
                inputSchema={
                    "type": "object",
                    "properties": {
                        "a": {"type": "number", "description": "First number"},
                        "b": {"type": "number", "description": "Second number"}
                    },
                    "required": ["a", "b"],
                },
            ),
            types.Tool(
                name="subtract",
                description="Subtract second number from first",
                inputSchema={
                    "type": "object",
                    "properties": {
                        "a": {"type": "number", "description": "First number"},
                        "b": {"type": "number", "description": "Second number"}
                    },
                    "required": ["a", "b"]
                },
            ),
        ]

    # ===========================================
    # Step 3: Implement Tool Handlers
    # ===========================================
    @server.call_tool()
    async def handle_call_tool(
        name: str, arguments: dict[str, Any] | None
    ) -> list[types.TextContent | types.ImageContent | types.EmbeddedResource]:
        """Handle tool execution requests."""
        try:
            if not arguments:
                raise ValueError("No arguments provided")
            
            if name == "add":
                if "a" not in arguments or "b" not in arguments:
                    raise ValueError("Missing required arguments: a, b")
                result = arguments['a'] + arguments['b']
                return [types.TextContent(type="text", text=str(result))]

            elif name == "subtract":
                if "a" not in arguments or "b" not in arguments:
                    raise ValueError("Missing required arguments: a, b")
                result = arguments['a'] - arguments['b']
                return [types.TextContent(type="text", text=str(result))]
            
            else:
                raise ValueError(f"Unknown tool: {name}")
                
        except Exception as e:
            logger.error(f"Error executing tool '{name}': {e}")
            return [types.TextContent(type="text", text=f"Error: {str(e)}")]

    # ===========================================
    # Step 4: Configure HTTP Session Manager
    # ===========================================
    session_manager = StreamableHTTPSessionManager(
        app=server,
        event_store=None,
        json_response=True,
        stateless=True,
    )

    # ===========================================
    # Step 5: Define Request Handler
    # ===========================================
    async def handle_streamable_http(
        scope: Scope, receive: Receive, send: Send
    ) -> None:
        """Handle incoming HTTP requests."""
        await session_manager.handle_request(scope, receive, send)

    # ===========================================
    # Step 6: Configure Application Lifecycle
    # ===========================================
    @contextlib.asynccontextmanager
    async def lifespan(app: Starlette) -> AsyncIterator[None]:
        """Manage application startup and shutdown."""
        async with session_manager.run():
            logger.info(f"✓ MCP Server started at http://127.0.0.1:{MCP_PORT}/mcp")
            try:
                yield
            finally:
                logger.info("✓ MCP Server shutting down...")

    # ===========================================
    # Step 7: Create and Run ASGI Application
    # ===========================================
    starlette_app = Starlette(
        debug=True,
        routes=[
            Mount("/mcp", app=handle_streamable_http),
        ],
        lifespan=lifespan,
    )

    # Start the server
    uvicorn.run(starlette_app, host="0.0.0.0", port=MCP_PORT)


# Entry point
if __name__ == "__main__":
    main()

In [ ]:
import subprocess
import time

# Start the HTTP server in the background
server_process = subprocess.Popen(
    ["python", "mcp_server_low_level_http.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for server to start
time.sleep(3)

# Store and display the process ID
SERVER_PID = server_process.pid
print(f"Process ID: {SERVER_PID}")

In [ ]:
!python mcp_client.py --http

In [ ]:
!kill -9 {SERVER_PID}
print(f"✓ Server (PID: {SERVER_PID}) killed")

### 参考リンク

- [uvicorn](https://github.com/Kludex/uvicorn)
- [starlette](https://github.com/Kludex/starlette)

---
## ライセンス

Copyright © 2026 OpenACC-Standard.org. 本資料は OpenACC-Standard.org が NVIDIA Corporation との協力のもと、Creative Commons Attribution 4.0 International (CC BY 4.0) ライセンスのもとで公開しています。本資料には他の団体が開発したハードウェアおよびソフトウェアへの参照が含まれており、該当するすべてのライセンスおよび著作権が適用されます。

<p> <center> <a href="../start_here_ja.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="02_introduction_mcp_ja.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="01_inference_endpoint_ja.ipynb">1</a>
        <a href="02_introduction_mcp_ja.ipynb">2</a>
        <a >3</a>
        <a href="04_langraph_agent_ja.ipynb">4</a>
        <a href="05_nemo_agent_toolkit_ja.ipynb">5</a>
        <a href="06_challenge_ja.ipynb">6</a>
        <a href="bonus_challenge/07_bonus_challenge_ja.ipynb">7</a>
    </span>
    <span style="float: left; width: 33%; text-align: right;"><a href="04_langraph_agent_ja.ipynb">Next Notebook</a></span>
</div>